In [1]:
import torch
print("="*50)
print("VÉRIFICATION GPU")
print("="*50)
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ GPU non détecté - arrêtez ici")

VÉRIFICATION GPU
CUDA disponible: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.3 GB


In [2]:
import os

data_yaml = "../data/data.yaml"

print("="*50)
print("VÉRIFICATION DONNÉES")
print("="*50)

if os.path.exists(data_yaml):
    print(f"✅ {data_yaml} trouvé")
else:
    print(f"❌ {data_yaml} non trouvé")

# Compter les images
train_images = len(os.listdir("../data/train/images")) if os.path.exists("../data/train/images") else 0
valid_images = len(os.listdir("../data/valid/images")) if os.path.exists("../data/valid/images") else 0
test_images = len(os.listdir("../data/test/images")) if os.path.exists("../data/test/images") else 0

print(f"📊 Train: {train_images} images")
print(f"📊 Valid: {valid_images} images") 
print(f"📊 Test: {test_images} images")

VÉRIFICATION DONNÉES
✅ ../data/data.yaml trouvé
📊 Train: 1054 images
📊 Valid: 123 images
📊 Test: 123 images


In [4]:
from ultralytics import YOLO

print("="*50)
print("ENTRAÎNEMENT BASELINE")
print("="*50)
print("⏱️ Durée estimée: 5-10 minutes")
print("")

# Charger modèle
model = YOLO('yolov8n.pt')
print("✅ Modèle yolov8n.pt chargé")

# Lancer l'entraînement
results = model.train(
    data=data_yaml,
    epochs=50,           # Très court - juste pour tester
    imgsz=416,           # Résolution réduite
    batch=8,             # Petit batch
    device=0,            # GPU
    workers=0,           # Simplifie pour Windows
    project="../models",
    name="baseline",
    exist_ok=True,
    verbose=True
)

print("")
print("✅ ENTRAÎNEMENT TERMINÉ!")

ENTRAÎNEMENT BASELINE
⏱️ Durée estimée: 5-10 minutes

✅ Modèle yolov8n.pt chargé
New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.162  Python-3.10.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, 

100%|██████████| 5.35M/5.35M [00:12<00:00, 453kB/s]


AMP: checks passed 
train: Fast image access  (ping: 0.10.1 ms, read: 41.814.3 MB/s, size: 15.6 KB)


train: Scanning C:\Users\ASUS\Documents\kidney-stone-detection\data\train\labels.cache... 1054 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1054/1054 [00:00<?, ?it/s]

val: Fast image access  (ping: 0.20.0 ms, read: 29.71.3 MB/s, size: 17.3 KB)



val: Scanning C:\Users\ASUS\Documents\kidney-stone-detection\data\valid\labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100%|██████████| 123/123 [00:00<?, ?it/s]


... (149 outputs left out)

                   all        123        325      0.753      0.597      0.638      0.251



50 epochs completed in 0.387 hours.
Optimizer stripped from ..\models\baseline\weights\last.pt, 6.2MB
Optimizer stripped from ..\models\baseline\weights\best.pt, 6.2MB

Validating ..\models\baseline\weights\best.pt...
Ultralytics 8.3.162  Python-3.10.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:01<00:00,  4.22it/s]


                   all        123        325      0.736      0.609      0.645      0.262
Speed: 0.2ms preprocess, 2.9ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to ..\models\baseline

✅ ENTRAÎNEMENT TERMINÉ!


In [7]:
import pandas as pd

csv_path = "../models/baseline/results.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    final_map50 = df['metrics/mAP50(B)'].iloc[-1]
    final_precision = df['metrics/precision(B)'].iloc[-1]
    final_recall = df['metrics/recall(B)'].iloc[-1]
    
    print("="*50)
    print("RÉSULTAT BASELINE")
    print("="*50)
    print(f"📊 mAP@50: {final_map50:.3f}")
    print(f"📊 Précision: {final_precision:.3f}")
    print(f"📊 Rappel: {final_recall:.3f}")
    
    if final_map50 > 0.1:
        print("\n✅ SUCCÈS! Le modèle apprend correctement.")
        print("→ Passez à l'étape suivante")
    else:
        print("\n⚠️ Résultat très faible. Vérifiez les données.")
else:
    print("❌ Fichier results.csv non trouvé")
    print("   L'entraînement a peut-être échoué")

RÉSULTAT BASELINE
📊 mAP@50: 0.638
📊 Précision: 0.753
📊 Rappel: 0.597

✅ SUCCÈS! Le modèle apprend correctement.
→ Passez à l'étape suivante


In [6]:
test_images_dir = "../data/test/images"
test_images_list = [f for f in os.listdir(test_images_dir) if f.endswith(('.jpg', '.png'))]

if test_images_list:
    test_img = test_images_list[0]
    results = model(f"{test_images_dir}/{test_img}")
    
    print(f"\n📸 Test sur: {test_img}")
    print(f"🔍 Détections: {len(results[0].boxes)}")
    
    # Afficher l'image (optionnel)
    from PIL import Image
    Image.open(f"{test_images_dir}/{test_img}").resize((300, 300))


image 1/1 c:\Users\ASUS\Documents\kidney-stone-detection\notebooks\..\data\test\images\1-3-46-670589-33-1-63703718086120120200001-5487554579919763006_png_jpg.rf.9fd67251e99a47dbe83a5db6efe6c016.jpg: 352x416 2 Tas_Vars, 55.7ms
Speed: 4.7ms preprocess, 55.7ms inference, 2.6ms postprocess per image at shape (1, 3, 352, 416)

📸 Test sur: 1-3-46-670589-33-1-63703718086120120200001-5487554579919763006_png_jpg.rf.9fd67251e99a47dbe83a5db6efe6c016.jpg
🔍 Détections: 2


In [ ]:
test_images_dir = "../data/test/images"
test_images_list = [f for f in os.listdir(test_images_dir) if f.endswith(('.jpg', '.png'))]

if test_images_list:
    test_img = test_images_list[0]
    results = model(f"{test_images_dir}/{test_img}")
    
    print(f"\n📸 Test sur: {test_img}")
    print(f"🔍 Détections: {len(results[0].boxes)}")
    
    # Afficher l'image (optionnel)
    from PIL import Image
    Image.open(f"{test_images_dir}/{test_img}").resize((300, 300))


image 1/1 c:\Users\ASUS\Documents\kidney-stone-detection\notebooks\..\data\test\images\1-3-46-670589-33-1-63703718086120120200001-5487554579919763006_png_jpg.rf.9fd67251e99a47dbe83a5db6efe6c016.jpg: 352x416 2 Tas_Vars, 55.7ms
Speed: 4.7ms preprocess, 55.7ms inference, 2.6ms postprocess per image at shape (1, 3, 352, 416)

📸 Test sur: 1-3-46-670589-33-1-63703718086120120200001-5487554579919763006_png_jpg.rf.9fd67251e99a47dbe83a5db6efe6c016.jpg
🔍 Détections: 2
